# Linha de base GPT autorregressivo em nível de caractere (nanoGPT)

**Projeto:** RNAP 2026/2 — Machado de Assis

Este notebook prepara os dados em caracteres, usa o código oficial do **nanoGPT** em um commit fixado e treina uma linha de base causal pequena do zero. A arquitetura segue o exemplo `shakespeare_char` do nanoGPT; o tokenizador é uma tabela de caracteres ajustada apenas no treino, com tokens reservados para fim de documento e caractere desconhecido. Portanto, não treinamos BPE nem afirmamos reproduzir o GPT-2 original em escala ou tokenização.

O enunciado da disciplina indica nanoGPT como código-base. O repositório agora se descreve como antigo/depreciado, então este notebook fixa o commit consultado e registra sua licença MIT. Na configuração padrão, o treino completo usa GPU; sem GPU, o notebook executa somente um teste curto de integração em CPU e deixa o treino completo para um ambiente Colab/GPU.

**Partições:** usamos `train.jsonl` e `validation.jsonl` durante o treino; `test.jsonl` só é aberto após existir o melhor checkpoint. A divisão por documento/obra foi feita no notebook 02. Mantemos cabeçalhos e notas editoriais conforme a preparação conservadora anterior.

Referências técnicas: [nanoGPT (README e código)](https://github.com/karpathy/nanoGPT), [GPT-2](https://openai.com/index/better-language-models/) e [Attention Is All You Need](https://arxiv.org/abs/1706.03762).

## Execução no Google Colab

1. Abra este notebook no Colab e escolha um runtime com GPU em **Runtime → Change runtime type**. A disponibilidade e o tipo de GPU variam.
2. Execute as células em ordem. Se os dados não estiverem na sessão, a célula 2 pedirá `train.jsonl`, `validation.jsonl` e `test.jsonl`; selecione-os em `projetos/machado-assis/dados/modelagem/`.
3. O treino completo será executado quando CUDA estiver disponível. Ao terminar, a exportação de artefatos é opcional na seção 6. Salve o ZIP ou copie a pasta para o Drive antes de encerrar a sessão, pois `/content` é temporário.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import pickle
import random
import subprocess
import sys

import numpy as np
import torch

PROJECT_DIR = Path.cwd()
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not (PROJECT_DIR / "projetos" / "machado-assis").exists():
    if PROJECT_DIR.name == "machado-assis":
        PROJECT_DIR = PROJECT_DIR.parent.parent
    elif IN_COLAB:
        # No Colab, arquivos e resultados ficam em /content durante a sessão.
        PROJECT_DIR = Path("/content")
    else:
        raise FileNotFoundError("Execute da raiz do workspace ou da pasta projetos/machado-assis.")
MACHADO_DIR = PROJECT_DIR / "projetos" / "machado-assis"
DATA_DIR = MACHADO_DIR / "dados"
SPLIT_DIR = DATA_DIR / "modelagem"
CHAR_DATA_DIR = SPLIT_DIR / "nanogpt_char"
EXPERIMENT_DIR = MACHADO_DIR / "experimentos" / "gpt_caractere_nanogpt"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CHAR_DATA_DIR.mkdir(parents=True, exist_ok=True)

NANOGPT_URL = "https://github.com/karpathy/nanoGPT.git"
NANOGPT_COMMIT = "3adf61e154c3fe3fca428ad6bc3818b27a3b8291"
# Repositório temporário, separado dos dados e dos resultados do projeto.
NANOGPT_DIR = Path(os.environ.get("TMPDIR", "/tmp")) / f"nanoGPT-machado-{NANOGPT_COMMIT[:12]}"
SEED = 20260925
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Sem CUDA: será feito apenas teste breve em CPU; o treino principal deve rodar em Colab/GPU.")
print("Dados preparados:", SPLIT_DIR)
print("Diretório dos resultados:", EXPERIMENT_DIR)


PyTorch: 2.13.0+cu130
Dispositivo: cpu
Sem CUDA: será feito apenas teste breve em CPU; o treino principal deve rodar em Colab/GPU.
Dados preparados: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/modelagem
Diretório dos resultados: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/experimentos/gpt_caractere_nanogpt


## 1. Obter e fixar o código-base nanoGPT

In [2]:
if not NANOGPT_DIR.exists():
    NANOGPT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", str(NANOGPT_DIR)], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(NANOGPT_DIR), "remote", "add", "origin", NANOGPT_URL], check=True)
    subprocess.run(["git", "-C", str(NANOGPT_DIR), "fetch", "--depth", "1", "origin", NANOGPT_COMMIT], check=True)
    subprocess.run(["git", "-C", str(NANOGPT_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)

actual_commit = subprocess.run(
    ["git", "-C", str(NANOGPT_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True
).stdout.strip()
assert actual_commit == NANOGPT_COMMIT, (actual_commit, NANOGPT_COMMIT)
license_path = NANOGPT_DIR / "LICENSE"
assert license_path.exists(), "Licença do código-base não encontrada."
print("nanoGPT commit:", actual_commit)
print("Licença presente:", license_path)
print("Código de treino:", NANOGPT_DIR / "train.py")


nanoGPT commit: 3adf61e154c3fe3fca428ad6bc3818b27a3b8291
Licença presente: /tmp/nanoGPT-machado-3adf61e154c3/LICENSE
Código de treino: /tmp/nanoGPT-machado-3adf61e154c3/train.py


## 2. Ajustar o tokenizador de caracteres somente no treino

In [3]:
split_paths = {name: SPLIT_DIR / f"{name}.jsonl" for name in ("train", "validation", "test")}
missing = [name for name, path in split_paths.items() if not path.exists()]
if missing and IN_COLAB:
    from google.colab import files
    print("Selecione train.jsonl, validation.jsonl e test.jsonl da pasta dados/modelagem do projeto.")
    uploaded = files.upload()
    for filename, contents in uploaded.items():
        if filename in {path.name for path in split_paths.values()}:
            (SPLIT_DIR / filename).write_bytes(contents)
    missing = [name for name, path in split_paths.items() if not path.exists()]
if missing:
    expected = ", ".join(path.name for path in split_paths.values())
    raise FileNotFoundError(
        f"Faltam {missing}. Coloque os arquivos {expected} em {SPLIT_DIR}; "
        "no Colab, a janela de upload aceita os três arquivos."
    )

def read_jsonl_documents(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

documents = {name: read_jsonl_documents(split_paths[name]) for name in ("train", "validation")}
assert documents["train"] and documents["validation"]

# O vocabulário é aprendido exclusivamente no treino. UNK cobre caracteres
# raros exclusivos da validação/teste; EOS marca o limite de cada arquivo.
train_characters = sorted(set("".join(doc["text"] for doc in documents["train"])))
EOS = "<|eos|>"
UNK = "<|unk|>"
itos = train_characters + [EOS, UNK]
stoi = {token: idx for idx, token in enumerate(itos)}
assert len(itos) <= 65536, "uint16 não comporta este vocabulário."
eos_id, unk_id = stoi[EOS], stoi[UNK]

def encode_documents(docs):
    ids = []
    unknown = 0
    for doc in docs:
        for char in doc["text"]:
            idx = stoi.get(char, unk_id)
            unknown += idx == unk_id
            ids.append(idx)
        ids.append(eos_id)
    return np.asarray(ids, dtype=np.uint16), unknown

encoded = {}
unknown_counts = {}
for split, docs in documents.items():
    encoded[split], unknown_counts[split] = encode_documents(docs)
    filename = "val.bin" if split == "validation" else "train.bin"
    encoded[split].tofile(CHAR_DATA_DIR / filename)
# Não deixamos um artefato de teste de execução anterior acessível ao treino.
test_bin_path = CHAR_DATA_DIR / "test.bin"
if test_bin_path.exists():
    test_bin_path.unlink()

meta = {
    "vocab_size": len(itos),
    "itos": itos,
    "stoi": stoi,
    "eos_token": EOS,
    "eos_id": eos_id,
    "unk_token": UNK,
    "unk_id": unk_id,
    "tokenizer": "character-level; vocabulary fitted on train documents only",
    "training_source": "train.jsonl",
    "seed": SEED,
}
with (CHAR_DATA_DIR / "meta.pkl").open("wb") as f:
    pickle.dump(meta, f)

print("Tamanho do vocabulário:", len(itos), "(caracteres + EOS + UNK)")
print("IDs de treino/validação:", {k: len(v) for k, v in encoded.items()})
print("Caracteres mapeados para UNK em treino/validação:", unknown_counts)
print("Dados binários em:", CHAR_DATA_DIR)
assert len(encoded["train"]) > 256 and len(encoded["validation"]) > 256


Tamanho do vocabulário: 147 (caracteres + EOS + UNK)
IDs de treino/validação: {'train': 11113894, 'validation': 1340703}
Caracteres mapeados para UNK em treino/validação: {'train': 0, 'validation': 2}
Dados binários em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/modelagem/nanogpt_char


## 3. Configurar o nanoGPT

A configuração principal espelha o modelo pequeno de caracteres documentado pelo nanoGPT (6 camadas, 6 cabeças, embedding 384, contexto de 256 caracteres). É uma linha de base educacional ajustada ao corpus, não uma reprodução do GPT‑2 original. Para cada execução registramos hiperparâmetros e semente.


In [4]:
OUT_DIR = EXPERIMENT_DIR / "out"
CONFIG_PATH = NANOGPT_DIR / "config" / "train_machado_char.py"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config_text = f'''# Configuração do projeto RNAP — derivada do padrão train_shakespeare_char do nanoGPT.
out_dir = r"{OUT_DIR}"
eval_interval = 500
log_interval = 50
eval_iters = 100
eval_only = False
always_save_checkpoint = True
init_from = "scratch"
dataset = "machado_assis_char"
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1
bias = False
learning_rate = 3e-4
max_iters = 5000
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0
decay_lr = True
warmup_iters = 100
lr_decay_iters = 5000
min_lr = 3e-5
backend = "nccl"
device = "cuda"
dtype = "float16"
compile = False
'''
CONFIG_PATH.write_text(config_text, encoding="utf-8")

# nanoGPT espera data/<dataset> dentro do clone; usa os .bin acima sem duplicar.
NANOGPT_DATA_DIR = NANOGPT_DIR / "data" / "machado_assis_char"
NANOGPT_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ("train.bin", "val.bin", "meta.pkl"):
    source = CHAR_DATA_DIR / name
    target = NANOGPT_DATA_DIR / name
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
print("Configuração:", CONFIG_PATH)
print("Pasta de dados lida pelo nanoGPT:", NANOGPT_DATA_DIR)
print(config_text)


Configuração: /tmp/nanoGPT-machado-3adf61e154c3/config/train_machado_char.py
Pasta de dados lida pelo nanoGPT: /tmp/nanoGPT-machado-3adf61e154c3/data/machado_assis_char
# Configuração do projeto RNAP — derivada do padrão train_shakespeare_char do nanoGPT.
out_dir = r"/workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/experimentos/gpt_caractere_nanogpt/out"
eval_interval = 500
log_interval = 50
eval_iters = 100
eval_only = False
always_save_checkpoint = True
init_from = "scratch"
dataset = "machado_assis_char"
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1
bias = False
learning_rate = 3e-4
max_iters = 5000
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0
decay_lr = True
warmup_iters = 100
lr_decay_iters = 5000
min_lr = 3e-5
backend = "nccl"
device = "cuda"
dtype = "float16"
compile = False



## 4. Teste de integração e treinamento

In [5]:
# Em CPU executamos apenas poucas iterações para provar o fluxo de dados/modelo.
# Com CUDA, o notebook executa o treinamento principal configurado acima.
RUN_FULL_TRAINING = torch.cuda.is_available()

if not RUN_FULL_TRAINING:
    smoke_out = Path("/tmp") / "machado_nanogpt_smoke"
    smoke_out.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "train.py", str(CONFIG_PATH),
        "--out_dir=" + str(smoke_out), "--device=cpu", "--dtype=float32", "--compile=False",
        "--max_iters=2", "--lr_decay_iters=2", "--warmup_iters=1",
        "--eval_interval=1", "--eval_iters=1", "--log_interval=1",
        "--batch_size=2", "--block_size=32", "--n_layer=2", "--n_head=2", "--n_embd=64",
        "--gradient_accumulation_steps=1", "--dropout=0.0",
    ]
    print("Teste curto de integração; não é resultado experimental nem checkpoint final.")
    subprocess.run(command, cwd=NANOGPT_DIR, check=True)
    print("Teste CPU concluído. Para o experimento, abra este notebook em Colab com GPU e execute novamente.")
else:
    command = [sys.executable, "train.py", str(CONFIG_PATH)]
    print("Iniciando treino completo em", DEVICE)
    subprocess.run(command, cwd=NANOGPT_DIR, check=True)
    checkpoint_path = OUT_DIR / "ckpt.pt"
    assert checkpoint_path.exists(), "nanoGPT terminou sem gerar o checkpoint esperado."
    print("Checkpoint final:", checkpoint_path)


Teste curto de integração; não é resultado experimental nem checkpoint final.


/tmp/nanoGPT-machado-3adf61e154c3/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


Overriding config with /tmp/nanoGPT-machado-3adf61e154c3/config/train_machado_char.py:
# Configuração do projeto RNAP — derivada do padrão train_shakespeare_char do nanoGPT.
out_dir = r"/workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/experimentos/gpt_caractere_nanogpt/out"
eval_interval = 500
log_interval = 50
eval_iters = 100
eval_only = False
always_save_checkpoint = True
init_from = "scratch"
dataset = "machado_assis_char"
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1
bias = False
learning_rate = 3e-4
max_iters = 5000
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0
decay_lr = True
warmup_iters = 100
lr_decay_iters = 5000
min_lr = 3e-5
backend = "nccl"
device = "cuda"
dtype = "float16"
compile = False

Overriding: out_dir = /tmp/machado_nanogpt_smoke
Overriding: device = cpu
Overriding: dtype = float32
Overriding: compile = False
Overriding: max_iters = 2
Overridin

Teste CPU concluído. Para o experimento, abra este notebook em Colab com GPU e execute novamente.


## 5. Avaliação final e amostras (após treino completo em GPU)

In [6]:
checkpoint_path = OUT_DIR / "ckpt.pt"
if not checkpoint_path.exists():
    print("Avaliação final ainda não executada: falta checkpoint do treino completo. Rode a seção 4 com GPU CUDA.")
else:
    if str(NANOGPT_DIR) not in sys.path:
        sys.path.insert(0, str(NANOGPT_DIR))
    from model import GPT, GPTConfig

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_args = checkpoint["model_args"]
    model = GPT(GPTConfig(**model_args)).to(device)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint["model"].items()}
    model.load_state_dict(state_dict)
    model.eval()
    block_size = model_args["block_size"]

    # O teste só é lido e codificado depois que o checkpoint foi produzido.
    test_documents = read_jsonl_documents(split_paths["test"])
    test_encoded, unknown_test_characters = encode_documents(test_documents)
    test_tokens = test_encoded.astype(np.int64)
    print("Documentos de teste abertos somente após o treino:", len(test_documents))
    rng = np.random.default_rng(SEED + 1)
    valid_starts = len(test_tokens) - block_size - 1
    assert valid_starts > 0
    eval_batches = 200
    batch_size = 32 if device.type == "cuda" else 8
    losses = []
    with torch.no_grad():
        for _ in range(eval_batches):
            starts = rng.integers(0, valid_starts, size=batch_size)
            x = torch.stack([torch.from_numpy(test_tokens[s:s+block_size]) for s in starts]).to(device)
            y = torch.stack([torch.from_numpy(test_tokens[s+1:s+block_size+1]) for s in starts]).to(device)
            _, loss = model(x, y)
            losses.append(float(loss.item()))
    test_loss = float(np.mean(losses))
    test_perplexity = float(np.exp(test_loss))

    # Geração controlada a partir de um prefixo genérico; EOS encerra a amostra.
    prompt = "Era "
    prompt_ids = [stoi.get(ch, unk_id) for ch in prompt]
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    torch.manual_seed(SEED)
    with torch.no_grad():
        generated = model.generate(idx, max_new_tokens=700, temperature=0.8, top_k=40)[0].tolist()
    generated = generated[len(prompt_ids):]
    if eos_id in generated:
        generated = generated[:generated.index(eos_id)]
    sample = "".join(itos[token] if token < len(itos)-2 else "�" for token in generated)
    sample_text = prompt + sample
    print(f"Test loss (nats/caractere, estimativa em {eval_batches} batches): {test_loss:.4f}")
    print(f"Perplexidade por caractere: {test_perplexity:.3f}")
    print("\nAmostra gerada (qualitativa; não é avaliação humana):\n")
    print(sample_text)

    results = {
        "nanoGPT_commit": NANOGPT_COMMIT,
        "seed": SEED,
        "model_args": model_args,
        "train_documents": len(documents["train"]),
        "validation_documents": len(documents["validation"]),
        "test_documents": len(test_documents),
        "vocab_size": len(itos),
        "unknown_test_characters": unknown_test_characters,
        "test_loss_nats_per_character_sampled": test_loss,
        "test_perplexity_per_character_sampled": test_perplexity,
        "evaluation_batches": eval_batches,
        "batch_size": batch_size,
        "sample_prompt": prompt,
        "sample_text": sample_text,
        "checkpoint": str(checkpoint_path),
    }
    (EXPERIMENT_DIR / "resultados_baseline.json").write_text(
        json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    (EXPERIMENT_DIR / "amostra_gerada.txt").write_text(sample_text + "\n", encoding="utf-8")
    print("Resultados registrados em:", EXPERIMENT_DIR)


Avaliação final ainda não executada: falta checkpoint do treino completo. Rode a seção 4 com GPU CUDA.


## 6. Exportar artefatos do Colab (opcional)

O disco de trabalho do Colab é temporário. Depois que a avaliação gerar o checkpoint, marque `EXPORT_COLAB_ARTIFACTS = True` e execute a célula seguinte para baixar um ZIP com pesos, tokenizer, configuração e resultados. Se não estiver no Colab, os arquivos ficam na pasta do projeto.


In [7]:
EXPORT_COLAB_ARTIFACTS = False

if IN_COLAB and EXPORT_COLAB_ARTIFACTS and (OUT_DIR / "ckpt.pt").exists():
    import shutil
    from google.colab import files
    shutil.copy2(CHAR_DATA_DIR / "meta.pkl", EXPERIMENT_DIR / "meta.pkl")
    shutil.copy2(CONFIG_PATH, EXPERIMENT_DIR / "config_train_machado_char.py")
    archive_base = Path("/content") / "machado_gpt_caractere_baseline"
    archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=EXPERIMENT_DIR))
    print("Baixando artefatos:", archive_path)
    files.download(str(archive_path))
elif IN_COLAB and EXPORT_COLAB_ARTIFACTS:
    print("Checkpoint não encontrado. Execute antes o treino completo e a avaliação.")
else:
    print("Exportação desligada. Em Colab, mude EXPORT_COLAB_ARTIFACTS para True para baixar os artefatos.")


Exportação desligada. Em Colab, mude EXPORT_COLAB_ARTIFACTS para True para baixar os artefatos.


## Interpretação e limites

- A perplexidade aqui é **por caractere**, não é diretamente comparável a valores BPE de artigos ou de GPT‑2.
- O modelo é treinado do zero num corpus autoral pequeno; não deve ser descrito como GPT‑2 pré-treinado nem como modelo capaz de responder perguntas.
- A amostra serve para inspeção qualitativa. No artigo, compare checkpoints/configurações sob as mesmas partições e reporte hiperparâmetros, seed, dispositivo, tempo e perdas de treino/validação/teste.
- A avaliação final deve ocorrer uma única vez depois da seleção pelo conjunto de validação; se os parâmetros forem ajustados com base no teste, este deixa de ser teste.
- O corpus continua sendo uma versão de trabalho, sem alegação de exaustividade bibliográfica.
